# 1. Introduction

This notebook prepares the demographic information contained in the `patients.csv` file from the processed Parkinson's Disease Smartwatch Dataset (PADS). The dataset includes participant identifiers, diagnostic labels, demographic characteristics, family-history information, and selected clinical variables.

The purpose of this notebook is to standardize categorical values, validate numerical variables, identify possible data-quality issues, and generate a cleaned demographic table for subsequent statistical analysis and machine learning.

## Objectives

This notebook aims to:

- Load the processed `patients.csv` dataset.
- Inspect the structure and completeness of the demographic data.
- Standardize categorical variables.
- Validate age, age at diagnosis, height, and weight.
- Create standardized diagnostic groups.
- Flag duplicate participant identifiers.
- Exclude sensitive free-text clinical comments.
- Export the cleaned demographic dataset and QA summary.


In [ ]:
from pathlib import Path
import pandas as pd

# 2. Loading the Demographic Data

The demographic data are loaded from the processed `patients.csv` file. The relative path below assumes that this notebook is stored in the `notebooks` folder and the dataset is stored in `data/processed`.


In [ ]:
# Define input and output paths
input_file = Path("../data/interim/patients.csv")
output_folder = Path("../data/interim")
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Input file: {input_file.resolve()}")
print(f"File exists: {input_file.exists()}")

if not input_file.exists():
    raise FileNotFoundError(
        f"Could not find {input_file}. "
        "Confirm that patients.csv is stored in data/processed."
    )

In [ ]:
# Load the demographic dataset
df = pd.read_csv(input_file)

print(f"Number of records: {len(df)}")
print(f"Number of columns: {df.shape[1]}")

df.head()

# 3. Initial Data Inspection

The dataset is reviewed to confirm its dimensions, variable names, data types, and missing values before cleaning.


In [ ]:
# Display column names and data types
df.info()

In [ ]:
# Summarize missing values
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_values")
)

missing_summary.head(15)

# 4. Standardizing Categorical Variables

Categorical variables are standardized to improve consistency. Gender and handedness values are cleaned by removing extra spaces and applying consistent capitalization. Family-history variables are converted to `Yes`, `No`, or `Unknown`, while missing alcohol-effect values are labelled as `Unknown`.


In [ ]:
# Preserve the original condition label for traceability
df["condition_original"] = df["condition"]

# Standardize gender and handedness
df["gender"] = df["gender"].astype("string").str.strip().str.title()
df["handedness"] = df["handedness"].astype("string").str.strip().str.title()

# Standardize Yes/No variables
yes_no_map = {
    True: "Yes",
    False: "No",
    "True": "Yes",
    "False": "No",
    "Yes": "Yes",
    "No": "No",
}

df["family_history_any"] = (
    df["appearance_in_kinship"]
    .map(yes_no_map)
    .fillna("Unknown")
)

df["family_history_first_degree"] = (
    df["appearance_in_first_grade_kinship"]
    .map(yes_no_map)
    .fillna("Unknown")
)

df["alcohol_effect_on_tremor"] = (
    df["effect_of_alcohol_on_tremor"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

# 5. Standardizing Diagnostic Groups

The numerical diagnosis labels are mapped to three standardized study groups: Healthy Control, Parkinson's Disease, and Other Movement Disorder.


In [ ]:
condition_map = {
    0: "Healthy Control",
    1: "Parkinson's Disease",
    2: "Other Movement Disorder",
}

df["condition_group"] = df["label"].map(condition_map)

df[["label", "condition_group"]].drop_duplicates().sort_values("label")

# 6. Validating Numerical Variables

Age, age at diagnosis, height, and weight are converted to numeric values. Values outside predefined plausible ranges are replaced with missing values. Age at diagnosis is also set to missing for healthy controls because it is not applicable.


In [ ]:
numeric_columns = [
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Replace implausible values with missing values
df.loc[~df["age"].between(18, 100), "age"] = pd.NA
df.loc[~df["height_cm"].between(120, 230), "height_cm"] = pd.NA
df.loc[~df["weight_kg"].between(30, 250), "weight_kg"] = pd.NA

# Diagnosis age is not applicable to healthy controls
df.loc[
    df["condition_group"] == "Healthy Control",
    "age_at_diagnosis"
] = pd.NA

# Remove invalid diagnosis ages
df.loc[
    (df["age_at_diagnosis"] <= 0)
    | (df["age_at_diagnosis"] > df["age"]),
    "age_at_diagnosis"
] = pd.NA

# 7. Quality Assurance Checks

Duplicate participant identifiers are flagged for review. Only the variables required for the cleaned demographic table are retained, while free-text clinical comments are excluded to reduce sensitivity and prevent possible target leakage.


In [ ]:
# Flag duplicate participant IDs
df["duplicate_patient_id"] = df["patient_id"].duplicated(keep=False)

columns_to_keep = [
    "patient_id",
    "study_id",
    "condition_original",
    "condition_group",
    "label",
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
    "gender",
    "handedness",
    "family_history_any",
    "family_history_first_degree",
    "alcohol_effect_on_tremor",
    "duplicate_patient_id",
]

clean_df = df[columns_to_keep].copy()

clean_df.head()

# 8. Cleaned Demographic Table

The cleaned demographic table contains standardized demographic and diagnostic variables suitable for subsequent statistical analysis and machine learning.


In [ ]:
# Display the cleaned demographic table
clean_df.head(10)

# 9. Exporting the Cleaned Data

The cleaned demographic table and a quality assurance summary are exported to the `data/processed` folder.


In [ ]:
# Save cleaned demographic table
clean_file = output_folder / "demographics_clean.csv"
clean_df.to_csv(clean_file, index=False)

# Create QA summary
summary = pd.DataFrame({
    "Check": [
        "Total records",
        "Unique participant IDs",
        "Duplicate participant records",
        "Missing age",
        "Missing age at diagnosis",
        "Missing height",
        "Missing weight",
    ],
    "Count": [
        len(clean_df),
        clean_df["patient_id"].nunique(),
        clean_df["duplicate_patient_id"].sum(),
        clean_df["age"].isna().sum(),
        clean_df["age_at_diagnosis"].isna().sum(),
        clean_df["height_cm"].isna().sum(),
        clean_df["weight_kg"].isna().sum(),
    ],
})

summary_file = output_folder / "demographics_qa_summary.csv"
summary.to_csv(summary_file, index=False)

print("Demographic cleaning completed.")
print(f"Clean data saved to: {clean_file}")
print(f"QA summary saved to: {summary_file}")

In [ ]:
# Display quality assurance summary
summary

# 10. Conclusion

The demographic data were cleaned by standardizing categorical variables, validating numerical values, creating consistent diagnostic groups, and flagging duplicate participant identifiers. Sensitive free-text comments were excluded from the modelling dataset.

The resulting `demographics_clean.csv` file provides a consistent demographic table for later analysis, while `demographics_qa_summary.csv` documents key quality-assurance results.
